In [2]:
import cv2
import numpy as np

img_dir = "../img/screen1.png"
img = cv2.imread(img_dir)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


import torchvision
from torchvision import models, datasets
import torchvision.transforms as T

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
model.eval()

#print(model.head)
category_names = [
    '__background__', 'person', 'car', 'wall', 'box',
]

COCO_INSTANCE_CATEGORY_NAMES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A', 'N/A',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table',
    'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'N/A', 'book',
    'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

transform = T.Compose([T.ToTensor()])
img = transform(img)
pred = model([img])
print(pred)

C:\Users\dkssu\AppData\Local\Programs\Python\Python39\lib\site-packages\torch\functional.py:445: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at  ..\aten\src\ATen\native\TensorShape.cpp:2157.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[{'boxes': tensor([[131.9748, 267.2519, 168.6185, 311.0847],
        [ 50.7987, 262.9058, 107.1792, 307.3260],
        [101.4329, 266.5389, 137.9459, 307.2961],
        [163.4865, 267.1005, 179.8960, 310.0618],
        [ 71.6562, 264.2196, 119.9872, 306.5465],
        [134.1919, 267.8674, 156.1792, 295.9661],
        [306.1396, 236.8707, 366.0703, 321.4010],
        [221.9040, 469.8361, 403.9333, 506.5626],
        [100.1413, 325.4276, 152.0639, 339.7596],
        [224.7729, 468.5792, 411.2982, 508.7443],
        [134.6396, 261.0279, 177.2663, 268.4812],
        [302.7506, 236.3088, 365.1883, 328.0611],
        [ 57.1327, 260.4422, 107.8848, 289.1707],
        [ 19.9316, 272.6252,  40.9823, 297.1066],
        [ 18.5997, 266.4991,  56.8169, 300.3801],
        [148.9776, 262.5287, 177.9562, 311.1394],
        [216.0392, 469.4460, 411.6902, 508.3355],
        [163.8577, 260.9293, 179.6215, 267.5285],
        [ 98.2980, 271.4985, 110.4617, 304.3487],
        [ 18.2731, 261.4639,  95.2529, 

In [3]:
import matplotlib.pyplot as plt

def get_prediction(img, threshold=0.7):
    #pred_class = [category_names[i] for i in list(pred[0]['labels'].numpy())]
    pred_class = [COCO_INSTANCE_CATEGORY_NAMES[i] for i in list(pred[0]['labels'].numpy())]
    pred_boxes = [[(i[0], i[1]), (i[2], i[3])] for i in list(pred[0]['boxes'].detach().numpy())]
    pred_score = list(pred[0]['scores'].detach().numpy())
    pred_thres = [pred_score.index(x) for x in pred_score if x>threshold][-1]
    pred_boxes = pred_boxes[:pred_thres+1]
    pred_class = pred_class[:pred_thres+1]

    return pred_boxes, pred_class

def object_detection_plt(img, threshold=0.5, rect_th=3, text_size=3, text_th=3):
    boxes, pred_cls = get_prediction(img, threshold)
    print(boxes)
    for i in range(len(boxes)):
        cv2.rectangle(img, boxes[i][0], boxes[i][1],color=(0, 255, 0), thickness=rect_th) # rectangle 사각형 그리기 함수, 시작점 좌표와 종료점 좌표를 기입하면 직각 사각형을 그릴 수 있다.
    cv2.putText(img,pred_cls[i], boxes[i][0], cv2.FONT_HERSHEY_SIMPLEX, text_size, (0,255,0),thickness=text_th) # puttext 문자 기입 함수, 글자 우측 하단을 시작점으로 하여 주어진 텍스트를 출력 
    plt.figure(figsize=(20,30))
    plt.imshow(img)
    plt.show()

In [4]:
img_dir = "../img/screen2.png"
img = cv2.imread(img_dir)

cv2.rectangle(img, (20,20),(40,40), (0,0,0), -1)

cv2.imshow("asd",img)
cv2.waitKey(10000)


-1

여기까지가 객체 인식,,

이를 통해 인식한 객체를 검은색 마스크로 씌워 라인 추적에 방해되지 않도록 만들기 

In [ ]:
img_dir = "../img/screen2.png"
img = cv2.imread(img_dir)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

object_detection_plt(img, threshold=0.8,rect_th=3, text_size=2, text_th=3)

In [17]:
import os
import torch
import torchvision
from torchvision import models, datasets, transforms
import torchvision.transforms as T

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


image_transforms = { 
    'train': transforms.Compose([
        transforms.RandomResizedCrop(size=256, scale=(0.8, 1.0)),
        transforms.RandomRotation(degrees=15), 
        transforms.RandomHorizontalFlip(), 
        transforms.CenterCrop(size=224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], # mean
                             [0.229, 0.224, 0.225]) # std
    ]),
    'valid': transforms.Compose([                   
        transforms.Resize(size=256),
        transforms.CenterCrop(size=224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(size=256),
        transforms.CenterCrop(size=224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
}


data_dir = "/d/github/video/src/img/track/"
train_dir = data_dir + "train/"
test_dir = data_dir + "test/"

batch_size = 4

COCO_INSTANCE_CATEGORY_NAMES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella',
    'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
    'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle',  'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
    'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 
    'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
    'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book',
    'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]

num_classes = [i for i in COCO_INSTANCE_CATEGORY_NAMES]
len_classes = len(num_classes)
print(len_classes)



model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)


91


In [23]:
from glob import glob
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torchvision
from torchvision import models, datasets
import torchvision.transforms as T

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)

os.makedirs("./weight",exist_ok=True)

torch.save(model, "./weight/model.pt")
dummy_input = torch.randn(1, 3, 224, 224,dtype=torch.float32)
#torch.onnx.export(model, dummy_input, dp_folder+"/weight/model.onnx",export_params=True, verbose=11)

<module 'torchvision.models' from 'C:\\Users\\dkssu\\AppData\\Local\\Programs\\Python\\Python39\\lib\\site-packages\\torchvision\\models\\__init__.py'>
